<center><b>EE (P) 533 Spring 2026<br>
Analog Circuits for Sensor Systems<br>
University of Washington Electrical & Computer Engineering</b></center>


<b>Instructor: Mahmood Hameed<br>
Assignment #3 (10 points)<br>
Due Monday, April 27 (Submit on Canvas as a Jupyter Notebook)</b> 

*Please show your work*

<b>Problem 1: Common-source amplifier with emitter-follower output stage

<center><img src="hw3.svg" width=650></center>


The MOSFET-based common-source amplifier can be combined with a BJT emitter-follower output stage to construct a circuit that provides voltage gain while taking advantage of the high input impedance of the MOSFET and low output impedance of the emitter follower.

For the following, $T = 27C$, $R_D = 5k\Omega$, $R_E = 2k\Omega$, $V_{IN,DC} = 0.9V$, $V_{DD} = 5V$, and $C_L = 1uF$

Use the Analog Devices MAT-02 $npn$ transistor ($V_A = 150V$, $\beta = 500$, $I_S = 0.6 pA$) for your analysis and simulations.

Use the Level 1 NMOS SPICE model provided with the assignment ($\mu = 350$, $C_{ox} = 384 nF/m^2$ $V_{th} = 0.7V$, $\lambda = 0.01 V^{-1}$, $W = 100\mu m$, $L = 1 \mu m$). 

<u>*Analysis*</u>

__a)__ *Estimate* the DC operating point of the circuit ($I_{D0}$, $I_{C0}$, $V_{1,DC}$, and $V_{OUT,DC}$) assuming $\lambda = 0$ for $M_1$ and $V_{BE} = 0.6V$, $I_B = 0$ for $Q_2$.

For the NMOS, I use the square-law saturation current with $\lambda=0$:

$$
K = \mu C_{ox}\frac{W}{L}, \qquad V_{OV}=V_{GS}-V_{th}
$$

$$
I_{D0}=\frac{1}{2}K V_{OV}^2.
$$

Here $V_{OV}=0.9-0.7=0.2V$. Also $V_{DS}\approx V_1$, so after the calculation I should check $V_{DS}>V_{OV}$ to make sure the NMOS is in saturation.

Since the problem says $I_B=0$, the BJT base does not load the drain node for this DC estimate. So

$$
V_{1,DC}=V_{DD}-I_{D0}R_D,
$$

$$
V_{OUT,DC}=V_{1,DC}-V_{BE}, \qquad I_{C0}\approx I_E=\frac{V_{OUT,DC}}{R_E}.
$$


In [1]:
import math

V_DD=5
R_D=5e3
R_E=2e3
V_IN_DC=0.9
V_th=0.7
V_BE=0.6

mu=350
C_ox=384e-9
W=100e-6
L=1e-6

V_ov=V_IN_DC-V_th
K=mu*C_ox*(W/L)
I_D0=0.5*K*V_ov**2
V1_DC=V_DD-I_D0*R_D
VOUT_DC=V1_DC-V_BE
I_C0=VOUT_DC/R_E

print(f'K={K} A/V^2')
print(f'V_ov={V_ov} V')
print(f'I_D0={I_D0*1e6} uA')
print(f'V1_DC={V1_DC} V')
print(f'VOUT_DC={VOUT_DC} V')
print(f'I_C0={I_C0*1e3} mA')


K=0.01344 A/V^2
V_ov=0.20000000000000007 V
I_D0=268.8000000000002 uA
V1_DC=3.655999999999999 V
VOUT_DC=3.0559999999999987 V
I_C0=1.5279999999999994 mA


So my estimated DC operating point is

$$
I_{D0}\approx 268.8\,\mu A, \qquad I_{C0}\approx 1.528\,mA,
$$

$$
V_{1,DC}\approx 3.656V, \qquad V_{OUT,DC}\approx 3.056V.
$$

The saturation check is also fine because $V_{DS}\approx3.656V > V_{OV}=0.2V$.


__b)__ Calculate the output resistance of the common-source stage (include $r_o$ of $M_1$), the input-resistance of the emitter-follower stage (*hint: this is not just $r_\pi$*!), and the output resistance of the emitter-follower (*hint: don't forget to use output resistance of common-source stage when calculating output resistance of emitter-follower.)

For the common-source stage output resistance, I include the MOSFET $r_o$:

$$
r_{o,M}=\frac{1}{\lambda I_{D0}}, \qquad R_{out,CS}=R_D \parallel r_{o,M}.
$$

For the emitter follower input resistance, the emitter sees $R_E \parallel r_{o,Q}$ to AC ground, so it is not just $r_\pi$:

$$
g_{m,Q}=\frac{I_{C0}}{V_T}, \qquad r_\pi=\frac{\beta}{g_{m,Q}}, \qquad r_{o,Q}=\frac{V_A}{I_{C0}}
$$

$$
R_{in2}=r_\pi+(\beta+1)(R_E\parallel r_{o,Q}).
$$

For the follower output resistance, I set the input source to small-signal ground. The base is then connected to ground through the common-source output resistance, so

$$
R_{out2}\approx R_E \parallel r_{o,Q} \parallel \frac{r_\pi+R_{out,CS}}{\beta+1}.
$$


In [2]:
V_T=25.85e-3
lambda_m=0.01
beta=500
V_A=150

r_o_m=1/(lambda_m*I_D0)
Rout_CS=(R_D*r_o_m)/(R_D+r_o_m)

g_m_m=K*V_ov
g_m_q=I_C0/V_T
r_pi=beta/g_m_q
r_o_q=V_A/I_C0

R_Ep=(R_E*r_o_q)/(R_E+r_o_q)
Rin2=r_pi+(beta+1)*R_Ep

Rbase=Rout_CS
R_small=(r_pi+Rbase)/(beta+1)
Rout2=1/(1/R_E+1/r_o_q+1/R_small)

print(f'g_m_M={g_m_m*1e3} mS')
print(f'r_o_M={r_o_m/1e3} kOhm')
print(f'Rout_CS={Rout_CS} Ohm')
print(f'g_m_Q={g_m_q*1e3} mS')
print(f'r_pi={r_pi} Ohm')
print(f'r_o_Q={r_o_q/1e3} kOhm')
print(f'Rin2={Rin2/1e3} kOhm')
print(f'Rout2={Rout2} Ohm')


g_m_M=2.688000000000001 mS
r_o_M=372.02380952380923 kOhm
Rout_CS=4933.69119040101 Ohm
g_m_Q=59.110251450676955 mS
r_pi=8458.769633507856 Ohm
r_o_Q=98.16753926701574 kOhm
Rin2=990.4522883456132 kOhm
Rout2=26.371799240297573 Ohm


The main results are

$$
R_{out,CS}\approx 4.93k\Omega,
$$

$$
R_{in2}\approx 990k\Omega,
$$

$$
R_{out2}\approx 26.4\Omega.
$$

This makes sense: the MOS stage has a few k$\Omega$ output resistance, the BJT stage has a high input resistance, and the emitter follower gives a small output resistance.


__c)__ Calculate the small-signal voltage gain and 3dB bandwidth $\left(f_{3dB} = \dfrac{\omega_0}{2 \pi} \right)$ of the circuit assuming $C_L$ is the only capacitance.

The first-stage gain is the common-source transconductance times the total drain resistance:

$$
g_{m,M}=K V_{OV}, \qquad R_{load1}=R_D\parallel r_{o,M}\parallel R_{in2}
$$

$$
A_{CS}=-g_{m,M}R_{load1}.
$$

The emitter follower gain is slightly less than 1:

$$
A_{EF}=\frac{g_{m,Q}+1/r_\pi}{g_{m,Q}+1/r_\pi+1/R_E+1/r_{o,Q}}.
$$

So the total low-frequency gain is

$$
A_v=A_{CS}A_{EF}.
$$

Since the only capacitance is $C_L$ at the output, I use the output resistance from part (b) for the pole:

$$
f_{3dB}=\frac{1}{2\pi R_{out2}C_L}.
$$


In [3]:
C_L=1e-6

Rload1=1/(1/R_D+1/r_o_m+1/Rin2)
A_cs=-g_m_m*Rload1
A_ef=(g_m_q+1/r_pi)/(g_m_q+1/r_pi+1/R_E+1/r_o_q)
A_v=A_cs*A_ef
A_v_dB=20*math.log10(abs(A_v))

f_3dB=1/(2*math.pi*Rout2*C_L)

print(f'Rload1=R_D||r_o_M||Rin2={Rload1} Ohm')
print(f'A_cs={A_cs} V/V')
print(f'A_ef={A_ef} V/V')
print(f'A_v={A_v} V/V')
print(f'A_v={A_v_dB} dB')
print(f'f_3dB={f_3dB} Hz')


Rload1=R_D||r_o_M||Rin2=4909.2370497332995 Ohm
A_cs=-13.196029189683115 V/V
A_ef=0.991459689948683 V/V
A_v=-13.083331008956993 V/V
A_v=22.33436658410264 dB
f_3dB=6035.043026138988 Hz


So the small-signal result is approximately

$$
A_v\approx -13.1\;V/V \qquad (22.3\,dB),
$$

and

$$
f_{3dB}\approx 6.0\,kHz.
$$

The negative sign comes from the common-source stage. The emitter follower mostly buffers the signal, so it does not flip the phase again.


<u>*Simulation*</u>

__d)__ Simulate the DC operating point (.op analysis) of the circuit in SPICE. Provide a screen capture of your circuit with all DC operating points labeled (right-click on the net and select *Place .op Data Label*).

Perform an AC simulation of the circuit and plot the frequency reponse. Indicate both the gain at $f = 10Hz$ and the 3dB bandwidth on the magnitude plot. 

Using either an AC analysis or DC Transfer analysis, simulate the DC input and output impedances of the circuit as discussed in Lecture 4.

I ran the LTspice simulations using the provided Level 1 NMOS model and a MAT02 BJT model. The hand calculation and simulation are close, but not exactly the same because SPICE includes finite base current, $\lambda$, body effect, and the exponential BJT equation instead of the simple $V_{BE}=0.6V$ estimate.

**.op simulation.** From the operating point simulation,

$$
V_{in}=0.900V, \qquad V_{DD}=5.000V,
$$

$$
V_1=3.591V, \qquad V_{out}=3.031V.
$$

The simulated device currents were

$$
I_D=0.2788\,mA, \qquad I_C=1.5127\,mA.
$$

**AC simulation.** With $V_{in}$ set to `AC 1`, the magnitude plot of $V(vout)$ directly gives the voltage gain. The measured low-frequency gain at 10 Hz was

$$
|A_v(10Hz)| = 22.65\,dB,
$$

and the 3dB bandwidth was

$$
f_{3dB}=6014\,Hz \approx 6.01\,kHz.
$$

**Input impedance simulation.** I measured the input impedance by applying an AC 1 V test source at the input and calculating $|V(vin)/I(Vin)|$ at 1 Hz. LTspice gave

$$
Z_{in}=225.427\,dB\Omega.
$$

Converting from dB,

$$
Z_{in}=10^{225.427/20}\approx1.87\times10^{11}\Omega.
$$

This is extremely large, which makes sense because the MOS gate draws almost no DC current.

**Output impedance simulation.** For output impedance, I set the input AC signal to 0 and applied an AC 1 A test current source at $vout$. Since the test current is 1 A, $|V(vout)|$ directly gives $Z_{out}$. LTspice gave

$$
Z_{out}=28.4528\,dB\Omega.
$$

So

$$
Z_{out}=10^{28.4528/20}\approx26.5\Omega.
$$

The simulated output resistance is very close to the hand calculation of about $26\Omega$.


*A few words on hand calculations versus simulation results: It is expected that your simulation results will not agree perfectly with your calculations. This is actually the purpose of using SPICE for verification, in that more detailed and more accurate models are employed to provide a more accurate prediction of circuit performance than the simple models used for "back-of-the-envelope" analysis.That being said, our simple models should gives us enough guidance on making design decisions, particularly if simulation results are different from what we expect.*